[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/SFM/blob/main/Quantlets/Ch_02/SFM_ch2_normal_distribution/SFM_ch2_normal_distribution.ipynb)

# SFM_ch2_normal_distribution

Normal Distribution and Normality Tests for Financial Returns
Description:
- PDF/CDF of the Normal distribution and its properties
- S&P 500 log-returns histogram with Normal overlay
- CLT illustration: convergence of t(3) sums to Normal
- Normality tests: Jarque-Bera, Shapiro-Wilk, Anderson-Darling
- QQ-plot of S&P 500 returns vs Normal distribution
Statistics of Financial Markets course

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
from scipy import stats
import os
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Chart style settings - Nature journal quality
plt.rcParams['figure.facecolor'] = 'none'
plt.rcParams['axes.facecolor'] = 'none'
plt.rcParams['savefig.facecolor'] = 'none'
plt.rcParams['savefig.transparent'] = True
plt.rcParams['axes.grid'] = False
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Helvetica', 'Arial', 'DejaVu Sans']
plt.rcParams['font.size'] = 8
plt.rcParams['axes.labelsize'] = 9
plt.rcParams['axes.titlesize'] = 10
plt.rcParams['xtick.labelsize'] = 8
plt.rcParams['ytick.labelsize'] = 8
plt.rcParams['legend.fontsize'] = 8
plt.rcParams['legend.facecolor'] = 'none'
plt.rcParams['legend.framealpha'] = 0
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.linewidth'] = 0.5
plt.rcParams['lines.linewidth'] = 0.75

# Named colors
MAIN_BLUE = '#1A3A6E'
CRIMSON   = '#DC3545'
FOREST    = '#2E7D32'
AMBER     = '#B5853F'
ORANGE    = '#E67E22'

import os
CHART_DIR = os.path.join('..', '..', '..', 'charts')
os.makedirs(CHART_DIR, exist_ok=True)

def save_fig(name):
    """Save figure with transparent background."""
    plt.savefig(os.path.join(CHART_DIR, f'{name}.pdf'),
                bbox_inches='tight', transparent=True)
    plt.savefig(os.path.join(CHART_DIR, f'{name}.png'),
                bbox_inches='tight', transparent=True, dpi=300)
    plt.close()
    print(f"   Saved: {name}.pdf/.png")

## Download Data

In [ ]:
ticker = '^GSPC'
data = yf.download(ticker, start='2000-01-01', end='2025-12-31', progress=False)
close = data['Close'].squeeze()

# Compute log-returns
log_returns = np.log(close / close.shift(1)).dropna()

print(f"   Ticker: {ticker} (S&P 500)")
print(f"   Period: {close.index[0].strftime('%Y-%m-%d')} to {close.index[-1].strftime('%Y-%m-%d')}")
print(f"   Observations: {len(close)}")
print(f"   Log-returns:  {len(log_returns)}")
print(f"")
print(f"   Summary Statistics of Log-Returns:")
print(f"     Mean:            {log_returns.mean():.6f}")
print(f"     Std Dev:         {log_returns.std():.6f}")
print(f"     Skewness:        {stats.skew(log_returns):.4f}")
print(f"     Excess Kurtosis: {stats.kurtosis(log_returns):.4f}")
print(f"     Min:             {log_returns.min():.6f}")
print(f"     Max:             {log_returns.max():.6f}")

## Normal Distribution PDF

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
fig.suptitle('Ch.2: Normal Distribution PDF', fontweight='bold', fontsize=12)

x = np.linspace(-6, 6, 500)

# Standard Normal N(0,1)
ax.plot(x, stats.norm.pdf(x, 0, 1), color=MAIN_BLUE, linewidth=2,
        label=r'$N(0,\,1)$ — Standard Normal')

# N(0, 2)
ax.plot(x, stats.norm.pdf(x, 0, 2), color=CRIMSON, linewidth=1.5,
        linestyle='--', label=r'$N(0,\,4)$ — $\mu=0,\,\sigma=2$')

# N(1, 0.5)
ax.plot(x, stats.norm.pdf(x, 1, 0.5), color=FOREST, linewidth=1.5,
        linestyle='-.', label=r'$N(1,\,0.25)$ — $\mu=1,\,\sigma=0.5$')

ax.set_xlabel('x')
ax.set_ylabel('Density f(x)')
ax.set_title('Normal (Gaussian) PDF with different parameters', fontweight='bold')
ax.legend(loc='upper right', frameon=False)

# Shade the standard normal between -1 and 1
x_fill = np.linspace(-1, 1, 200)
ax.fill_between(x_fill, stats.norm.pdf(x_fill, 0, 1),
                alpha=0.15, color=MAIN_BLUE)
ax.annotate(r'68.3% of area', xy=(0, 0.15), fontsize=8,
            ha='center', color=MAIN_BLUE)

plt.tight_layout()
save_fig('ch2_normal_pdf')

## S&P 500 Histogram with Normal Fit

In [ ]:
mu = log_returns.mean()
sigma = log_returns.std()

fig, ax = plt.subplots(figsize=(10, 5))
fig.suptitle('Ch.2: S&P 500 Log-Returns with Normal Fit', fontweight='bold', fontsize=12)

# Histogram
ax.hist(log_returns, bins=100, density=True, alpha=0.5, color=MAIN_BLUE,
        edgecolor='white', linewidth=0.3, label='S&P 500 log-returns')

# Normal MLE overlay
x = np.linspace(log_returns.min() * 1.1, log_returns.max() * 1.1, 500)
ax.plot(x, stats.norm.pdf(x, mu, sigma), color=CRIMSON, linewidth=2,
        label=f'Normal fit: $\\mu$={mu:.5f}, $\\sigma$={sigma:.5f}')

# Mark mean +/- 1, 2, 3 sigma
for k, ls, alpha in [(1, '--', 0.8), (2, '-.', 0.6), (3, ':', 0.4)]:
    ax.axvline(mu + k * sigma, color=FOREST, linestyle=ls, linewidth=1, alpha=alpha)
    ax.axvline(mu - k * sigma, color=FOREST, linestyle=ls, linewidth=1, alpha=alpha)
    y_pos = stats.norm.pdf(mu + k * sigma, mu, sigma) * 1.15
    ax.text(mu + k * sigma, y_pos, f'+{k}$\\sigma$', ha='center', fontsize=7,
            color=FOREST, alpha=alpha + 0.1)
    ax.text(mu - k * sigma, y_pos, f'-{k}$\\sigma$', ha='center', fontsize=7,
            color=FOREST, alpha=alpha + 0.1)

# Annotate sample statistics
skew_val = stats.skew(log_returns)
kurt_val = stats.kurtosis(log_returns)
stats_text = (f'n = {len(log_returns)}\n'
              f'$\\mu$ = {mu:.5f}\n'
              f'$\\sigma$ = {sigma:.5f}\n'
              f'Skewness = {skew_val:.4f}\n'
              f'Ex. Kurtosis = {kurt_val:.4f}')
ax.text(0.97, 0.95, stats_text, transform=ax.transAxes, ha='right', va='top',
        fontsize=8, bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

ax.set_xlabel('Log-Return')
ax.set_ylabel('Density')
ax.set_title('Histogram of Daily Log-Returns with Normal Overlay', fontweight='bold')
ax.legend(loc='upper left', frameon=False)

plt.tight_layout()
save_fig('ch2_normal_fit')

## QQ-Plot

In [ ]:
# Standardize returns
z_returns = (log_returns - log_returns.mean()) / log_returns.std()

fig, ax = plt.subplots(figsize=(7, 7))
fig.suptitle('Ch.2: QQ-Plot of S&P 500 Returns vs Normal', fontweight='bold', fontsize=12)

# QQ-plot
osm, osr = stats.probplot(z_returns, dist='norm', fit=False)
ax.scatter(osm, osr, s=8, alpha=0.5, color=MAIN_BLUE, edgecolors='none',
           label='S&P 500 standardized returns')

# 45-degree reference line
lim = max(abs(min(osm.min(), osr.min())), abs(max(osm.max(), osr.max()))) * 1.05
ax.plot([-lim, lim], [-lim, lim], color=CRIMSON, linewidth=1.5,
        linestyle='--', label='45-degree line (perfect Normal)')

# Annotate heavy tails
ax.annotate('Heavy left tail\n(more extreme losses\nthan Normal predicts)',
            xy=(osm[2], osr[2]),
            xytext=(-2.5, osr.min() * 0.6),
            fontsize=8, color=CRIMSON,
            arrowprops=dict(arrowstyle='->', color=CRIMSON, lw=1),
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

ax.annotate('Heavy right tail\n(more extreme gains\nthan Normal predicts)',
            xy=(osm[-3], osr[-3]),
            xytext=(2.0, osr.max() * 0.6),
            fontsize=8, color=CRIMSON,
            arrowprops=dict(arrowstyle='->', color=CRIMSON, lw=1),
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

ax.set_xlabel('Theoretical Quantiles (Normal)')
ax.set_ylabel('Sample Quantiles (S&P 500)')
ax.set_title('Normal QQ-Plot of S&P 500 Daily Log-Returns', fontweight='bold')
ax.legend(loc='upper left', frameon=False)
ax.set_xlim(-lim, lim)
ax.set_ylim(-lim, lim)
ax.set_aspect('equal')

plt.tight_layout()
save_fig('ch2_qqplot')

## Normality Tests

In [ ]:
# --- Jarque-Bera test ---
jb_stat, jb_pval = stats.jarque_bera(log_returns)

# --- Shapiro-Wilk test (cap at 5000 observations) ---
sw_sample = log_returns[:5000] if len(log_returns) > 5000 else log_returns
sw_stat, sw_pval = stats.shapiro(sw_sample)

# --- Anderson-Darling test ---
ad_result = stats.anderson(log_returns, dist='norm')
ad_stat = ad_result.statistic
# Use 5% significance level
ad_cv_5pct = ad_result.critical_values[2]  # index 2 = 5% level
ad_reject = 'Yes' if ad_stat > ad_cv_5pct else 'No'

# Print formatted results
print("   " + "="*60)
print("   NORMALITY TESTS FOR S&P 500 LOG-RETURNS")
print("   " + "="*60)
print(f"   {'Test':<25} {'Statistic':>12} {'p-value':>12} {'Reject H0?':>12}")
print("   " + "-"*60)
print(f"   {'Jarque-Bera':<25} {jb_stat:>12.2f} {jb_pval:>12.6f} {'Yes' if jb_pval < 0.05 else 'No':>12}")
print(f"   {'Shapiro-Wilk':<25} {sw_stat:>12.6f} {sw_pval:>12.6f} {'Yes' if sw_pval < 0.05 else 'No':>12}")
print(f"   {'Anderson-Darling':<25} {ad_stat:>12.4f} {'cv=' + str(round(ad_cv_5pct, 4)):>12} {ad_reject:>12}")
print("   " + "="*60)
print("   H0: Data is normally distributed (alpha = 0.05)")
print(f"   Shapiro-Wilk uses first {len(sw_sample)} observations (max 5000)")

# --- Visual summary ---
fig, ax = plt.subplots(figsize=(8, 3))
fig.suptitle('Ch.2: Normality Tests Summary', fontweight='bold', fontsize=12)
ax.axis('off')

table_data = [
    ['Jarque-Bera', f'{jb_stat:.2f}', f'{jb_pval:.2e}', 'Yes' if jb_pval < 0.05 else 'No'],
    ['Shapiro-Wilk', f'{sw_stat:.6f}', f'{sw_pval:.2e}', 'Yes' if sw_pval < 0.05 else 'No'],
    ['Anderson-Darling', f'{ad_stat:.4f}', f'cv={ad_cv_5pct:.4f}', ad_reject]
]

table = ax.table(
    cellText=table_data,
    colLabels=['Test', 'Statistic', 'p-value / CV', 'Reject H\u2080 (5%)'],
    cellLoc='center',
    loc='center',
    colColours=[MAIN_BLUE]*4
)

table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1, 1.8)

# Style header row
for j in range(4):
    table[0, j].set_text_props(color='white', fontweight='bold')
    table[0, j].set_facecolor(MAIN_BLUE)

# Style data rows - highlight rejections in red
for i in range(1, 4):
    for j in range(4):
        table[i, j].set_facecolor('white')
        table[i, j].set_edgecolor('#CCCCCC')
    if table_data[i-1][3] == 'Yes':
        table[i, 3].set_text_props(color=CRIMSON, fontweight='bold')

ax.text(0.5, -0.05, 'H\u2080: Data is normally distributed | S&P 500 daily log-returns',
        transform=ax.transAxes, ha='center', fontsize=8, color='gray')

plt.tight_layout()
save_fig('ch2_normality_tests')

## Central Limit Theorem Illustration

In [ ]:
np.random.seed(42)

n_sims = 100_000
sample_sizes = [1, 5, 30, 100]
df_t = 3  # t-distribution with 3 degrees of freedom

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
fig.suptitle('Ch.2: Central Limit Theorem — Sums of i.i.d. t(3) Variables',
             fontweight='bold', fontsize=12)

for idx, n in enumerate(sample_sizes):
    ax = axes[idx // 2, idx % 2]

    # Generate n_sims sums of n i.i.d. t(3) random variables
    samples = stats.t.rvs(df=df_t, size=(n_sims, n))
    sums = samples.sum(axis=1)

    # Standardize: E[X] = 0 for t, Var[X] = df/(df-2) = 3 for t(3)
    std_sums = sums / np.sqrt(n * df_t / (df_t - 2))

    # Histogram
    ax.hist(std_sums, bins=100, density=True, alpha=0.5, color=MAIN_BLUE,
            edgecolor='white', linewidth=0.3, label=f'Standardized sum (n={n})')

    # N(0,1) overlay
    x = np.linspace(-5, 5, 300)
    ax.plot(x, stats.norm.pdf(x, 0, 1), color=CRIMSON, linewidth=2,
            label='N(0, 1)')

    # KS test
    ks_stat, ks_pval = stats.kstest(std_sums, 'norm')
    ax.text(0.97, 0.95, f'KS stat = {ks_stat:.4f}\np-value = {ks_pval:.2e}',
            transform=ax.transAxes, ha='right', va='top', fontsize=8,
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

    ax.set_title(f'n = {n}', fontweight='bold')
    ax.set_xlabel('Standardized Sum')
    ax.set_ylabel('Density')
    ax.set_xlim(-5, 5)
    ax.legend(loc='upper left', frameon=False, fontsize=7)

plt.tight_layout(rect=[0, 0, 1, 0.95])
save_fig('ch2_clt_illustration')